# Native CLM Architecture Lab — Phase C

Phase C crosses only mechanisms that already won independently:

- **X1** — N1 residual-only + SwiGLU
- **X2** — N1 residual-only + RoPE
- **X3** — N1 residual-only + SwiGLU + RoPE
- **X4** — X3 + RMSNorm interaction re-test

C0/T1/C1/M4/N1 are frozen anchors. The runner also produces reproducible SVG comparisons for final PPL, learning curves, and quality/compute.


In [ ]:
BRANCH = "research/native-clm-lab"
PROFILE = "baseline"
SEED = 91001
PHASE_C_MODELS = [
    "X1-n1-swiglu",
    "X2-n1-rope",
    "X3-n1-swiglu-rope",
    "X4-n1-swiglu-rope-rmsnorm",
]
RUN_PHASE_C = True
ALLOW_CPU = False
PUSH_DEV_RESULT = True
assert SEED == 91001, "Phase C must not consume held-back seeds"


## Environment bootstrap

Existing Kaggle checkouts are fast-forwarded before execution. Large caches/checkpoints stay under `/kaggle/working/native-clm`.


In [ ]:
from pathlib import Path
import base64, json, os, shutil, subprocess, sys

def run_checked(cmd, **kwargs):
    return subprocess.run(cmd, check=True, text=True, **kwargs)

repo_candidates = [Path.cwd(), Path("/kaggle/working/mini-cells")]
REPO_ROOT = next((p for p in repo_candidates if (p / ".git").exists()), None)
if REPO_ROOT is None:
    REPO_ROOT = Path("/kaggle/working/mini-cells")
    run_checked(["git","clone","--depth","1","--branch",BRANCH,
                 "https://github.com/ArcheLabs/mini-cells.git",str(REPO_ROOT)])
elif Path("/kaggle/working").exists():
    run_checked(["git","fetch","origin",BRANCH], cwd=REPO_ROOT)
    run_checked(["git","switch",BRANCH], cwd=REPO_ROOT)
    run_checked(["git","pull","--ff-only","origin",BRANCH], cwd=REPO_ROOT)

run_checked([sys.executable,"-m","pip","install","-q","-e",str(REPO_ROOT)+"[lm]"])
WORK_ROOT = Path("/kaggle/working/native-clm") if Path("/kaggle/working").exists() else REPO_ROOT/".native-clm-work"
CACHE_ROOT, OUTPUT_ROOT, HF_ROOT = WORK_ROOT/"cache", WORK_ROOT/"runs", WORK_ROOT/"huggingface"
for p in (CACHE_ROOT, OUTPUT_ROOT, HF_ROOT):
    p.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_ROOT))
os.environ.setdefault("HF_DATASETS_CACHE", str(HF_ROOT/"datasets"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(HF_ROOT/"hub"))
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("repo:", REPO_ROOT)
print("HEAD:", subprocess.check_output(["git","rev-parse","--short","HEAD"], cwd=REPO_ROOT, text=True).strip())


In [ ]:
def read_secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

HF_TOKEN = read_secret("HF_TOKEN")
GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
print({"HF_TOKEN_available": bool(HF_TOKEN), "GITHUB_TOKEN_available": bool(GITHUB_TOKEN)})


## Phase-C protocol gates

Before any 10M-token run, instantiate all candidates, verify parameter matching, run a finite forward smoke test, and run the repository's Phase-C tests.


In [ ]:
sys.path.insert(0, str(REPO_ROOT/"research/native-clm"))
import torch
import native_clm_runtime as base
from native_clm_phase_c import PHASE_C_NAMES, build_phase_c, parameter_summary, phase_c_config, estimate_flops

params = parameter_summary()
for name in PHASE_C_MODELS:
    assert name in PHASE_C_NAMES
    assert params[name]["relative_error"] < 0.01

ids = torch.randint(0, base.VOCAB_SIZE, (2, 16))
for name in PHASE_C_MODELS:
    model = build_phase_c(name)
    with torch.no_grad():
        logits = model(ids)
    assert logits.shape == (2, 16, base.VOCAB_SIZE)
    assert torch.isfinite(logits).all()
    del model

run_checked([sys.executable,"-m","pytest","-q",
             str(REPO_ROOT/"research/native-clm/test_native_clm_phase_c.py")], cwd=REPO_ROOT)

audit = {
    "parameters": params,
    "configs": {n: phase_c_config(n) for n in PHASE_C_MODELS},
    "train_flops_estimate": {
        n: estimate_flops(n, base.PROFILES[PROFILE].target_tokens)["train_flops_estimate"]
        for n in PHASE_C_MODELS
    },
}
print(json.dumps(audit, indent=2))


## Run Phase C

With two GPUs this runs two rounds: X1/X2, then X3/X4. Frozen anchors are read from committed evidence and are not retrained.


In [ ]:
RUNNER = REPO_ROOT/"research/native-clm/run_native_clm_phase_c.py"
cmd = [
    sys.executable, str(RUNNER), "sweep",
    "--models", *PHASE_C_MODELS,
    "--profile", PROFILE,
    "--seed", str(SEED),
    "--cache-root", str(CACHE_ROOT),
    "--output-root", str(OUTPUT_ROOT),
]
if ALLOW_CPU:
    cmd.append("--allow-cpu")
print("launching Phase C:", PHASE_C_MODELS)
if RUN_PHASE_C:
    run_checked(cmd, cwd=REPO_ROOT)
else:
    print("RUN_PHASE_C=False; existing outputs only")


## Leaderboard and visual comparison

The figures below are generated from the exact JSON/CSV evidence. They are also committed as SVG files.


In [ ]:
RUN_DIR = OUTPUT_ROOT/f"phase-c-{PROFILE}-seed-{SEED}"
LEADERBOARD = RUN_DIR/"phase-c-leaderboard.json"
if not LEADERBOARD.is_file():
    raise FileNotFoundError(LEADERBOARD)
rows = json.loads(LEADERBOARD.read_text(encoding="utf-8"))

try:
    import pandas as pd
    display(pd.DataFrame(rows).sort_values("validation_ppl_10m"))
except Exception:
    print(json.dumps(sorted(rows, key=lambda r:r["validation_ppl_10m"]), indent=2))

from IPython.display import SVG, display
for filename in (
    "phase-c-final-ppl.svg",
    "phase-c-learning-curves.svg",
    "phase-c-quality-compute.svg",
):
    print(filename)
    display(SVG(filename=str(RUN_DIR/filename)))


## Record and push development evidence

Only small summaries, checkpoint curves, JSON/CSV and the generated SVGs enter Git. Authentication failure can no longer turn a successful experiment into a notebook exception: without `GITHUB_TOKEN`, the evidence remains as a local commit and push is skipped.


In [ ]:
record_dir = REPO_ROOT/"research/native-clm/results/dev"/f"phase-c-{PROFILE}-seed-{SEED}"
record_dir.mkdir(parents=True, exist_ok=True)

for name in (
    "protocol.json",
    "phase-c-leaderboard.json",
    "phase-c-leaderboard.csv",
    "phase-c-summary.json",
    "phase-c-visualizations.json",
    "phase-c-final-ppl.svg",
    "phase-c-learning-curves.svg",
    "phase-c-quality-compute.svg",
):
    src = RUN_DIR/name
    if src.is_file():
        shutil.copy2(src, record_dir/name)

for model in PHASE_C_MODELS:
    model_id = model.split("-")[0]
    for src_name, dst_name in (
        ("summary.json", f"{model_id}-summary.json"),
        ("checkpoints.csv", f"{model_id}-checkpoints.csv"),
    ):
        src = RUN_DIR/model/src_name
        if src.is_file():
            shutil.copy2(src, record_dir/dst_name)

ranked = sorted(rows, key=lambda r:r["validation_ppl_10m"])
readme = "# Native CLM Phase-C development evidence\n\n"
readme += "\n".join(
    f"- {r['id']}: PPL={r['validation_ppl_10m']:.6f}, params={r['parameters']}, "
    f"FLOPs={r['train_flops_estimate']:.3e}"
    for r in ranked
)
readme += "\n\nVisualizations: `phase-c-final-ppl.svg`, `phase-c-learning-curves.svg`, `phase-c-quality-compute.svg`.\n"
readme += "\nSeed 91001 only; development evidence, not a promoted confirmation result.\n"
(record_dir/"README.md").write_text(readme, encoding="utf-8")
print("development evidence:", record_dir)


In [ ]:
def commit_and_push_evidence(repo, record_dir, branch, token):
    relative = record_dir.relative_to(repo)
    run_checked(["git","config","user.name","native-clm-kaggle"], cwd=repo)
    run_checked(["git","config","user.email","native-clm@users.noreply.github.com"], cwd=repo)
    run_checked(["git","add",str(relative)], cwd=repo)
    changed = subprocess.run(["git","diff","--cached","--quiet"], cwd=repo).returncode != 0
    if changed:
        run_checked(["git","commit","-m",f"research: record Native CLM Phase-C seed {SEED}"], cwd=repo)
    else:
        print("no new evidence to commit")
    head = subprocess.check_output(["git","rev-parse","HEAD"], cwd=repo, text=True).strip()
    if not token:
        print("GITHUB_TOKEN unavailable; evidence committed locally only:", head)
        return
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    env["GIT_CONFIG_COUNT"] = "1"
    env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
    env["GIT_CONFIG_VALUE_0"] = f"AUTHORIZATION: basic {basic}"
    run_checked(
        ["git","push","https://github.com/ArcheLabs/mini-cells.git",f"HEAD:{branch}"],
        cwd=repo, env=env,
    )
    print("pushed Phase-C evidence:", head)

if PUSH_DEV_RESULT:
    commit_and_push_evidence(REPO_ROOT, record_dir, BRANCH, GITHUB_TOKEN)
else:
    print("automatic evidence push disabled")
